<a href="https://colab.research.google.com/github/itsPronay/ai_hub_benchmark/blob/main/Test_on_gpu_cpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/itsPronay/ai_hub_benchmark

Cloning into 'ai_hub_benchmark'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 78 (delta 31), reused 54 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 40.13 KiB | 8.03 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [2]:
import sys
sys.path.append('/content/ai_hub_benchmark')

In [3]:
import argparse
from model.vit import ViT
import torch
import time
import numpy as np

In [4]:
parser = argparse.ArgumentParser(description='Colab-benchmark-ViT')

parser.add_argument('--patches', type=int, default=15)
parser.add_argument('--device_to_test', type=str, default='cuda')
parser.add_argument('--mode', choices=['CAF', 'ViT'], default='ViT')
parser.add_argument('--num_classes', type=int, default=9)
parser.add_argument('--band', type=int, default=30)
parser.add_argument('--wandb_project', default='Edge-benchmark_of_vit')
parser.add_argument('--wandb_mode',   default='disabled', choices=['online', 'offline', 'disabled'])

args, _ = parser.parse_known_args()

In [5]:
def benchmark_model(model, input_tensor, runs, run_warmup=True, warmup_runs=10):

  model.eval()

  #warmup
  with torch.no_grad():
    if run_warmup:
      for _ in range(warmup_runs):
        _ = model(input_tensor)

  #sync before timing (run on gpu only)
  if input_tensor.device.type == 'cuda':
    torch.cuda.synchronize()

  latencies = []

  for _ in range(runs):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(input_tensor)
    if input_tensor.device.type == 'cuda':
        torch.cuda.synchronize()
    latencies.append((time.perf_counter() - start) * 1000)


  batch_size = input_tensor.shape[0] # 1000, 30, 225 -> Batch, band, patch * patch
  mean_latency_ms = float(np.mean(latencies))
  min_latency_ms = float(np.min(latencies))
  max_latency_ms = float(np.max(latencies))
  std = float(np.std(latencies))

  result = {
      # 'device : ''
      'input_shape' : str(input_tensor.shape),
      'mean_latency_ms' : round(mean_latency_ms, 4),
      'min_latency_ms' : round(min_latency_ms, 4),
      'max_latency_ms' : round(max_latency_ms, 4),
      'std_latency_ms' : round(max_latency_ms, 4),
  }

  return result


In [7]:
def main():

  input_shapes = [
      1, # 1 x 1
      100, # 10 x 10
      250, # 25 x 10
      500, # 50 x 10
      1000, # 100 x 10
      1500, # 150 x 10
      # 2000, # 200 x 10
      # 4000, # 400 x 10
      # 8000, # 800 x 10
      # 16000 # 1600 x 10
  ]

  devices_to_test = ['cpu']
  if torch.cuda.is_available():
      devices_to_test.append('cuda')
      print(f"GPU detected: {torch.cuda.get_device_name(0)}")
  else:
      print("No GPU found – CPU only.")

  model = ViT(
      image_size   = args.patches,
      near_band    = 1,
      num_patches  = args.band,
      num_classes  = args.num_classes,
      dim          = 64,
      depth        = 5,
      heads        = 4,
      mlp_dim      = 8,
      dropout      = 0.1,
      emb_dropout  = 0.1,
      mode         = args.mode
  ).eval()


  all_results = []

  for batch in input_shapes:
    shape = (batch, args.band, args.patches * args.patches)
    print(f"Running for input shape: {shape}")

    for device in devices_to_test:
      model = model.to(device)
      input_tensor = torch.randn(shape, device=device)
      metrics = benchmark_model(model, input_tensor, runs=10)

      metrics.update({
          'device' : device,
          'mode' : args.mode,
          'spatial_patch_size' : args.patches,
          'spectral_bands' : 1,
          'num_classes' : args.num_classes,
          'batch_size' : batch,
       })

      all_results.append(metrics)



  print(f"\n{'='*80}")
  print(f"{'Shape':<30} {'Device':<6} {'Mean(ms)':>10} {'Std(ms)':>15}")
  print('-'*80)
  for r in all_results:
    if r['device'] == 'cuda':
      print(f"{r['input_shape']:<30} {r['device']:<6} "
            f"{r['mean_latency_ms']:>8} {r['std_latency_ms']:>15} ")

  print()
  print()

  print(f"\n{'='*80}")
  print(f"{'Shape':<30} {'Device':<6} {'Mean(ms)':>10} {'Std(ms)':>15} ")
  print('-'*80)
  for r in all_results:
    if r['device'] == 'cpu':
      print(f"{r['input_shape']:<30} {r['device']:<6} "
            f"{r['mean_latency_ms']:>8} {r['std_latency_ms']:>15} ")

if __name__ == '__main__':
    main()


GPU detected: Tesla T4
Running for input shape: (1, 30, 225)
Running for input shape: (100, 30, 225)
Running for input shape: (250, 30, 225)
Running for input shape: (500, 30, 225)
Running for input shape: (1000, 30, 225)
Running for input shape: (1500, 30, 225)

Shape                          Device   Mean(ms)         Std(ms)
--------------------------------------------------------------------------------
torch.Size([1, 30, 225])       cuda     3.3892          3.5997 
torch.Size([100, 30, 225])     cuda     4.0126          4.8082 
torch.Size([250, 30, 225])     cuda     6.6074          6.6506 
torch.Size([500, 30, 225])     cuda     9.9313         10.0093 
torch.Size([1000, 30, 225])    cuda     17.075         21.9824 
torch.Size([1500, 30, 225])    cuda    22.7871         24.9771 



Shape                          Device   Mean(ms)         Std(ms) 
--------------------------------------------------------------------------------
torch.Size([1, 30, 225])       cpu      3.0455          